# Near-Parallel Features on Real Data (Longley, 1947-1962)

The same instability from the airline solver, but inside an ordinary **linear regression**. We pull the Longley economic dataset live from the internet — the textbook multicollinearity case where `GNP`, `Population`, and `Year` all rise together.

In [1]:
import numpy as np, pandas as pd, requests
from io import StringIO
from sklearn.linear_model import LinearRegression
np.set_printoptions(precision=4, suppress=True)

URL = ('https://vincentarelbundock.github.io/'
       'Rdatasets/csv/datasets/longley.csv')
df = pd.read_csv(StringIO(requests.get(URL, timeout=30).text))
df = df.drop(columns=['rownames'])
df.head()

,GNP.deflator,GNP,Unemployed,Armed.Forces,Population,Year,Employed
0,83.0,234.289,235.6,159.0,107.608,1947,60.323
1,88.5,259.426,232.5,145.6,108.632,1948,61.122
2,88.2,258.054,368.2,161.6,109.773,1949,60.171
3,89.5,284.599,335.1,165.0,110.929,1950,61.187
4,96.2,328.975,209.9,309.9,112.075,1951,63.221


In [2]:
FEATURES = ['GNP.deflator','GNP','Unemployed',
            'Armed.Forces','Population','Year']
TARGET = 'Employed'
X = df[FEATURES].to_numpy(float)
y = df[TARGET].to_numpy(float)
print('X shape:', X.shape, '(only 16 rows -> the bug is LOUD)')

X shape: (16, 6) (only 16 rows -> the bug is LOUD)


## 1. A feature is a vector — are any two nearly parallel?

Center each column (compare *shape*, not the fact they're all positive), then measure the angle. `cos → 1` means parallel.

In [3]:
Xc = X - X.mean(axis=0)
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"{'pair':<28}{'cos':>10}{'angle(deg)':>12}")
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        c = cosine(Xc[:, i], Xc[:, j])
        if abs(c) > 0.99:
            ang = np.degrees(np.arccos(np.clip(c, -1, 1)))
            print(f'{FEATURES[i]+" vs "+FEATURES[j]:<28}{c:>10.4f}{ang:>12.2f}')

pair                               cos  angle(deg)
GNP.deflator vs GNP             0.9916        7.44
GNP.deflator vs Year            0.9911        7.63
GNP vs Population               0.9911        7.65
GNP vs Year                     0.9953        5.57
Population vs Year              0.9940        6.30


## 2. Condition number — how ill-conditioned is the whole matrix?

Rule of thumb: `>1e3` shaky, `>1e6` dangerous.

In [4]:
Xdesign = np.column_stack([np.ones(len(X)), X])
cond = np.linalg.cond(Xdesign)
print(f'cond(X) = {cond:,.0f}')
print(f'~ you lose about {np.log10(cond):.0f} of ~16 decimal digits')

cond(X) = 23,845,862
~ you lose about 7 of ~16 decimal digits


## 3. QR pre-check — *which* columns are the culprits?

Small `|diag(R)|` = that column adds almost no new direction.

In [5]:
Q, R = np.linalg.qr(Xdesign)
diag = np.abs(np.diag(R))
names = ['intercept'] + FEATURES
for n, d in sorted(zip(names, diag), key=lambda t: t[1]):
    flag = '  <-- redundant direction' if d < 0.1*diag.max() else ''
    print(f'{n:<16}|R_ii| = {d:>12.4f}{flag}')

Year            |R_ii| =       0.6693  <-- redundant direction
Population      |R_ii| =       1.4632  <-- redundant direction
intercept       |R_ii| =       4.0000  <-- redundant direction
GNP.deflator    |R_ii| =      41.7955
GNP             |R_ii| =      49.8229
Armed.Forces    |R_ii| =     170.3533
Unemployed      |R_ii| =     282.0602


## 4. The payoff — nudge the data 0.1%, watch coefficients move

Standardize first so every coefficient is on the same scale. The condition number *is* the amplification factor.

In [6]:
Xz = (X - X.mean(0)) / X.std(0)
rng = np.random.default_rng(0)

def fit_coefs(Xmat, jitter=0.0):
    if jitter:
        Xmat = Xmat + rng.normal(0, jitter, Xmat.shape)
    return LinearRegression().fit(Xmat, y).coef_

before = fit_coefs(Xz)
after  = fit_coefs(Xz, jitter=0.001)
swing = np.linalg.norm(after-before)/np.linalg.norm(before)
print(f"{'feature':<16}{'clean':>12}{'nudged':>12}{'abs change':>12}")
for n,a,b in zip(FEATURES, before, after):
    print(f'{n:<16}{a:>12.3f}{b:>12.3f}{abs(b-a):>12.3f}')
print(f'\noverall coefficient vector moved {swing:.0%} from a 0.1% nudge')

feature                clean      nudged  abs change
GNP.deflator           0.157       0.149       0.009
GNP                   -3.447      -3.435       0.013
Unemployed            -1.828      -1.832       0.004
Armed.Forces          -0.696      -0.703       0.007
Population            -0.344      -0.383       0.039
Year                   8.432       8.472       0.040

overall coefficient vector moved 1% from a 0.1% nudge


## 5. The fix — drop the near-parallel columns

Keep one trend (`Year`), drop the rest (`GNP`, `Population`, `GNP.deflator`). The condition number collapses and the model becomes stable.

In [7]:
KEEP = ['Unemployed', 'Armed.Forces', 'Year']
ki = [FEATURES.index(k) for k in KEEP]
Xkz = Xz[:, ki]

cond_b = np.linalg.cond(np.column_stack([np.ones(len(Xz)), Xz]))
cond_a = np.linalg.cond(np.column_stack([np.ones(len(Xkz)), Xkz]))
print(f'cond (standardized) BEFORE : {cond_b:,.0f}')
print(f'cond (standardized) AFTER  : {cond_a:,.1f}')

c0 = fit_coefs(Xkz)
c1 = fit_coefs(Xkz, jitter=0.001)
sw = np.linalg.norm(c1-c0)/np.linalg.norm(c0)
print(f"\n{'feature':<16}{'clean':>12}{'nudged':>12}{'abs change':>12}")
for n,a,b in zip(KEEP, c0, c1):
    print(f'{n:<16}{a:>12.3f}{b:>12.3f}{abs(b-a):>12.3f}')
print(f'\noverall coefficient vector moved only {sw:.1%} now')

cond (standardized) BEFORE : 111
cond (standardized) AFTER  : 3.7

feature                clean      nudged  abs change
Unemployed            -1.330      -1.332       0.002
Armed.Forces          -0.520      -0.521       0.001
Year                   4.409       4.410       0.002

overall coefficient vector moved only 0.1% now


### Read it

Same data, same algorithm. The only change is **removing the duplicate directions you can now see**. `cond` falls from ~111 to ~3.7 and the model stops lying.